# Lesson 6 — Bending Moment Diagram

**Course:** Python for Structural Engineers — Simply Supported Beam  
**Prerequisites:** Lessons 1–5  
**You will learn:**
- Integrate the shear force diagram to obtain the bending moment diagram
- Use `scipy.integrate.cumulative_trapezoid` for numerical integration
- Plot sagging (positive) and hogging (negative) regions with colour fills
- Build the complete three-panel interactive widget: beam + SFD + BMD

**Time estimate:** 60–75 minutes

---

## The Theory — Moment as the Integral of Shear

The fundamental relationship between shear force V and bending moment M is:

$$\frac{dM}{dx} = V(x)$$

Which means:

$$M(x) = M(0) + \int_0^x V(\xi) \, d\xi$$

Since the left end of the beam is free (no moment applied), **M(0) = 0**.

**Key rules for the BMD shape:**
- Where V is **constant**, M changes **linearly**
- Where V is **linearly varying** (under UDL), M is **parabolic**
- Where V = 0, M is at a **local maximum or minimum**
- At the **free ends** of cantilevers, M = 0
- At the **pin and roller supports** (A and B), the moment is generally **not zero** — the beam is continuous through the supports

⚠️ **Important:** A simply supported beam with cantilevers has zero moment at the free tips, NOT at the supports. The supports provide vertical force only, not moment resistance.

---

## 6.1 Numerical Integration with SciPy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid
import sys
sys.path.insert(0, "../shared")
import beam_solver
from plot_helpers import draw_beam, draw_loads, draw_reactions

# Beam and loads (same as previous lessons)
geometry = {"L_total": 10.0, "x_A": 2.0, "x_B": 8.0}
loads    = [
    {"type": "point", "x": 5.0,  "magnitude": -20.0},
    {"type": "udl",   "x1": 2.0, "x2": 8.0, "w":  -5.0},
]

# Step 1: reactions and shear
R_A, R_B = beam_solver.compute_reactions(geometry, loads)
x, V     = beam_solver.compute_shear(geometry, loads, R_A, R_B)

print(f"R_A = {R_A:+.2f} kN,  R_B = {R_B:+.2f} kN")
print(f"V[0]  = {V[0]:+.4f} kN  (free left end, should be ≈ 0)")
print(f"V[-1] = {V[-1]:+.4f} kN  (free right end, should be ≈ 0)")

In [ ]:
# Step 2: integrate V to get M
# cumulative_trapezoid integrates step by step: M[i] = M[i-1] + area under V between x[i-1] and x[i]

M = cumulative_trapezoid(V, x, initial=0.0)

print(f"M[0]  = {M[0]:+.4f} kN·m  (should be 0 — free left end)")
print(f"M[-1] = {M[-1]:+.4f} kN·m  (should be ≈ 0 — free right end)")
print(f"M_max = {M.max():+.2f} kN·m  at x = {x[M.argmax()]:.2f} m")
print(f"M_min = {M.min():+.2f} kN·m  at x = {x[M.argmin()]:.2f} m")

**Self-check:** `M[-1]` should be ≈ 0 (free right end). If it is not, the reactions are wrong.

---

## 6.2 Plotting the BMD

Convention: **sagging positive** (blue fill), **hogging negative** (red fill).

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))

# The BMD line
ax.plot(x, M, color="#1B5E20", linewidth=2.2, label="M(x)")

# Sagging (positive) region — blue
ax.fill_between(x, M, 0,
                where=(M >= 0), color="#A5D6A7", alpha=0.55, label="Sagging (+)")

# Hogging (negative) region — red
ax.fill_between(x, M, 0,
                where=(M < 0),  color="#FFCDD2", alpha=0.55, label="Hogging (−)")

# Zero line
ax.axhline(0, color="black", linewidth=0.8)

# Mark peak moment
idx_max = M.argmax()
idx_min = M.argmin()
ax.plot(x[idx_max], M[idx_max], "o", color="#1B5E20", markersize=8, zorder=5)
ax.annotate(f"M_max = {M[idx_max]:+.1f} kN·m\nx = {x[idx_max]:.2f} m",
            xy=(x[idx_max], M[idx_max]),
            xytext=(x[idx_max] + 0.4, M[idx_max] - 5),
            fontsize=8.5, color="#1B5E20",
            arrowprops=dict(arrowstyle="->", color="#1B5E20", lw=0.8))

if abs(M[idx_min]) > 0.5:
    ax.plot(x[idx_min], M[idx_min], "o", color="#C62828", markersize=8, zorder=5)
    ax.annotate(f"M_min = {M[idx_min]:+.1f} kN·m\nx = {x[idx_min]:.2f} m",
                xy=(x[idx_min], M[idx_min]),
                xytext=(x[idx_min] + 0.4, M[idx_min] + 5),
                fontsize=8.5, color="#C62828",
                arrowprops=dict(arrowstyle="->", color="#C62828", lw=0.8))

ax.set_ylabel("Bending Moment M (kN·m)", fontsize=10)
ax.set_xlabel("Position x (m)", fontsize=10)
ax.set_xlim(x[0] - 0.2, x[-1] + 0.2)
ax.legend(fontsize=8, loc="upper right")
ax.grid(True, linestyle="--", alpha=0.4)
ax.set_title("Bending Moment Diagram", fontsize=11, pad=6)

plt.tight_layout()
plt.show()

---

## 6.3 The Three-Panel Figure

The most useful view — beam, SFD, and BMD stacked vertically with a shared x-axis:

In [ ]:
def plot_full_analysis(geometry, loads):
    """Draw the complete three-panel analysis figure."""
    R_A, R_B = beam_solver.compute_reactions(geometry, loads)
    x, V     = beam_solver.compute_shear(geometry, loads, R_A, R_B)
    M        = beam_solver.compute_moments(x, V)

    fig = plt.figure(figsize=(14, 10))
    gs  = fig.add_gridspec(3, 1, height_ratios=[1, 1.8, 1.8], hspace=0.08)

    ax_beam = fig.add_subplot(gs[0])
    ax_sfd  = fig.add_subplot(gs[1])
    ax_bmd  = fig.add_subplot(gs[2])

    # ── Beam ────────────────────────────────────────────────────────────────
    draw_beam(ax_beam, geometry)
    draw_loads(ax_beam, loads, geometry["L_total"])
    draw_reactions(ax_beam, geometry["x_A"], geometry["x_B"], R_A, R_B)

    # ── SFD ─────────────────────────────────────────────────────────────────
    ax_sfd.plot(x, V, color="#1565C0", lw=2.0)
    ax_sfd.fill_between(x, V, 0, where=(V>=0), color="#90CAF9", alpha=0.5)
    ax_sfd.fill_between(x, V, 0, where=(V<0),  color="#EF9A9A", alpha=0.5)
    ax_sfd.axhline(0, color="black", lw=0.8)
    ax_sfd.set_ylabel("V (kN)", fontsize=9)
    ax_sfd.grid(True, ls="--", alpha=0.35)
    ax_sfd.set_title("Shear Force Diagram", fontsize=9, loc="left", pad=3)

    # Mark V=0 crossings on SFD
    zc = np.where(np.diff(np.sign(V)))[0]
    for idx in zc:
        ax_sfd.axvline(x[idx], color="red", ls=":", lw=0.9, alpha=0.7)
        ax_bmd.axvline(x[idx], color="red", ls=":", lw=0.9, alpha=0.7)

    # ── BMD ─────────────────────────────────────────────────────────────────
    ax_bmd.plot(x, M, color="#1B5E20", lw=2.0)
    ax_bmd.fill_between(x, M, 0, where=(M>=0), color="#A5D6A7", alpha=0.5,
                        label="Sagging (+)")
    ax_bmd.fill_between(x, M, 0, where=(M<0),  color="#FFCDD2", alpha=0.5,
                        label="Hogging (−)")
    ax_bmd.axhline(0, color="black", lw=0.8)
    ax_bmd.set_ylabel("M (kN·m)", fontsize=9)
    ax_bmd.set_xlabel("Position x (m)", fontsize=10)
    ax_bmd.legend(fontsize=8, loc="upper right")
    ax_bmd.grid(True, ls="--", alpha=0.35)
    ax_bmd.set_title("Bending Moment Diagram", fontsize=9, loc="left", pad=3)

    # Annotate M peaks
    for idx, color in [(M.argmax(), "#1B5E20"), (M.argmin(), "#C62828")]:
        if abs(M[idx]) > 0.5:
            ax_bmd.plot(x[idx], M[idx], "o", color=color, ms=7, zorder=5)
            ax_bmd.annotate(f"{M[idx]:+.1f}",
                            xy=(x[idx], M[idx]), xytext=(x[idx]+0.3, M[idx]),
                            fontsize=8, color=color)

    # Shared x-limits
    xlim = (x[0] - 0.3, x[-1] + 0.3)
    for ax in [ax_beam, ax_sfd, ax_bmd]:
        ax.set_xlim(*xlim)

    # Remove x-tick labels from upper panels
    ax_beam.tick_params(labelbottom=False)
    ax_sfd.tick_params(labelbottom=False)

    fig.suptitle(
        f"Simply Supported Beam Analysis\n"
        f"L={geometry['L_total']} m  |  A at {geometry['x_A']} m  |  "
        f"B at {geometry['x_B']} m  |  "
        f"R_A={R_A:+.1f} kN  |  R_B={R_B:+.1f} kN",
        fontsize=10, y=1.01
    )
    plt.tight_layout()
    plt.show()


plot_full_analysis(geometry, loads)

Read the three panels together:
- The **red dashed lines** mark where V = 0, which is exactly where M peaks
- The **slope of the BMD** at any point equals the shear value at that point
- Both cantilever tips (x = 0 and x = 10 m) have M = 0

---

## 6.4 Full Interactive Three-Panel Widget

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.insert(0, "../shared")
import beam_solver
from plot_helpers import draw_beam, draw_loads, draw_reactions

%matplotlib inline

live_loads = []
out_plot   = widgets.Output()
out_status = widgets.Output()

style  = {"description_width": "170px"}
layout = widgets.Layout(width="460px")

sl_L  = widgets.FloatSlider(value=10.0, min=5.0, max=20.0, step=0.5,
                             description="Total length L (m):", style=style, layout=layout)
sl_xA = widgets.FloatSlider(value=2.0,  min=0.0, max=9.5,  step=0.5,
                             description="Support A at (m):",   style=style, layout=layout)
sl_xB = widgets.FloatSlider(value=8.0,  min=0.5, max=10.0, step=0.5,
                             description="Support B at (m):",   style=style, layout=layout)
dd_type = widgets.Dropdown(options=[("Point load","point"),("UDL","udl")],
                            description="Load type:", style=style, layout=layout)
sl_x    = widgets.FloatSlider(value=5.0, min=0.0, max=10.0, step=0.5,
                               description="Position x (m):", style=style, layout=layout)
sl_P    = widgets.FloatSlider(value=-20.0, min=-150.0, max=150.0, step=5.0,
                               description="Magnitude (kN):", style=style, layout=layout)
sl_x1   = widgets.FloatSlider(value=2.0, min=0.0, max=9.5, step=0.5,
                               description="UDL start x1 (m):", style=style, layout=layout)
sl_x2   = widgets.FloatSlider(value=8.0, min=0.5, max=10.0, step=0.5,
                               description="UDL end x2 (m):",   style=style, layout=layout)
sl_w    = widgets.FloatSlider(value=-5.0, min=-40.0, max=40.0, step=1.0,
                               description="Intensity (kN/m):", style=style, layout=layout)
chk_peaks = widgets.Checkbox(value=True, description="Annotate peak values")
chk_fill  = widgets.Checkbox(value=True, description="Colour fills on SFD/BMD")

btn_add   = widgets.Button(description="Add Load",  button_style="primary")
btn_clear = widgets.Button(description="Clear All", button_style="danger")
pt_ctrl   = widgets.VBox([sl_x, sl_P])
udl_ctrl  = widgets.VBox([sl_x1, sl_x2, sl_w])
udl_ctrl.layout.display = "none"

def on_type(c):
    if c["new"]=="point":
        pt_ctrl.layout.display=""; udl_ctrl.layout.display="none"
    else:
        pt_ctrl.layout.display="none"; udl_ctrl.layout.display=""
dd_type.observe(on_type, names="value")

def redraw(change=None):
    L   = sl_L.value
    x_A = min(sl_xA.value, sl_xB.value - 0.5)
    x_B = min(sl_xB.value, L)
    geo = {"L_total": L, "x_A": x_A, "x_B": x_B}

    with out_plot:
        out_plot.clear_output(wait=True)
        if not live_loads:
            fig, ax = plt.subplots(figsize=(13, 3.5))
            draw_beam(ax, geo)
            ax.set_title("Add loads to compute SFD and BMD", fontsize=10, color="gray")
            plt.tight_layout(); plt.show(); return

        try:
            R_A, R_B = beam_solver.compute_reactions(geo, live_loads)
            x, V     = beam_solver.compute_shear(geo, live_loads, R_A, R_B)
            M        = beam_solver.compute_moments(x, V)
        except Exception as e:
            print(f"Error: {e}"); return

        fig = plt.figure(figsize=(14, 10.5))
        gs  = fig.add_gridspec(3, 1, height_ratios=[1, 1.8, 1.8], hspace=0.08)
        ax_beam = fig.add_subplot(gs[0])
        ax_sfd  = fig.add_subplot(gs[1])
        ax_bmd  = fig.add_subplot(gs[2])

        draw_beam(ax_beam, geo)
        draw_loads(ax_beam, live_loads, L)
        draw_reactions(ax_beam, x_A, x_B, R_A, R_B)

        # SFD
        ax_sfd.plot(x, V, color="#1565C0", lw=2.0)
        if chk_fill.value:
            ax_sfd.fill_between(x, V, 0, where=(V>=0), color="#90CAF9", alpha=0.5)
            ax_sfd.fill_between(x, V, 0, where=(V<0),  color="#EF9A9A", alpha=0.5)
        ax_sfd.axhline(0, color="black", lw=0.8)
        ax_sfd.set_ylabel("V (kN)", fontsize=9)
        ax_sfd.grid(True, ls="--", alpha=0.35)

        # BMD
        ax_bmd.plot(x, M, color="#1B5E20", lw=2.0)
        if chk_fill.value:
            ax_bmd.fill_between(x, M, 0, where=(M>=0), color="#A5D6A7", alpha=0.5,
                                label="Sagging (+)")
            ax_bmd.fill_between(x, M, 0, where=(M<0),  color="#FFCDD2", alpha=0.5,
                                label="Hogging (−)")
            ax_bmd.legend(fontsize=8, loc="upper right")
        ax_bmd.axhline(0, color="black", lw=0.8)
        ax_bmd.set_ylabel("M (kN·m)", fontsize=9)
        ax_bmd.set_xlabel("Position x (m)", fontsize=10)
        ax_bmd.grid(True, ls="--", alpha=0.35)

        # V=0 dashed lines
        zc = np.where(np.diff(np.sign(V)))[0]
        for idx in zc:
            ax_sfd.axvline(x[idx], color="red", ls=":", lw=0.9, alpha=0.7)
            ax_bmd.axvline(x[idx], color="red", ls=":", lw=0.9, alpha=0.7)

        if chk_peaks.value:
            for idx, color, ax_ in [(M.argmax(),"#1B5E20",ax_bmd),
                                    (M.argmin(),"#C62828",ax_bmd),
                                    (V.argmax(),"#0D47A1",ax_sfd),
                                    (V.argmin(),"#B71C1C",ax_sfd)]:
                arr = M if ax_ is ax_bmd else V
                if abs(arr[idx]) > 0.5:
                    ax_.plot(x[idx], arr[idx], "o", color=color, ms=6, zorder=5)
                    ax_.annotate(f"{arr[idx]:+.1f}",
                                 xy=(x[idx], arr[idx]),
                                 xytext=(x[idx]+0.25, arr[idx]),
                                 fontsize=8, color=color)

        xlim = (x[0]-0.3, x[-1]+0.3)
        for ax_ in [ax_beam, ax_sfd, ax_bmd]:
            ax_.set_xlim(*xlim)
        ax_beam.tick_params(labelbottom=False)
        ax_sfd.tick_params(labelbottom=False)

        fig.suptitle(
            f"L={L:.1f} m  |  A at {x_A:.1f} m  |  B at {x_B:.1f} m  "
            f"|  R_A={R_A:+.1f} kN  |  R_B={R_B:+.1f} kN",
            fontsize=10, y=1.01
        )
        plt.tight_layout(); plt.show()

for sl in [sl_L, sl_xA, sl_xB, chk_peaks, chk_fill]:
    sl.observe(redraw, names="value")

def on_add(b):
    with out_status:
        out_status.clear_output(wait=True)
        if dd_type.value=="point":
            live_loads.append({"type":"point","x":sl_x.value,"magnitude":sl_P.value})
            print(f"✓ {sl_P.value:+.0f} kN at x={sl_x.value:.1f} m")
        else:
            if sl_x2.value<=sl_x1.value:
                print("Error: x2 must be > x1"); return
            live_loads.append({"type":"udl","x1":sl_x1.value,
                                "x2":sl_x2.value,"w":sl_w.value})
            print(f"✓ UDL {sl_w.value:+.0f} kN/m  [{sl_x1.value:.1f}–{sl_x2.value:.1f} m]")
    redraw()

def on_clear(b):
    live_loads.clear()
    with out_status:
        out_status.clear_output(wait=True); print("Cleared.")
    redraw()

btn_add.on_click(on_add)
btn_clear.on_click(on_clear)

redraw()
display(widgets.VBox([
    widgets.HTML("<b>Geometry</b>"),
    sl_L, sl_xA, sl_xB,
    widgets.HTML("<hr><b>Add Loads</b>"),
    dd_type, pt_ctrl, udl_ctrl,
    widgets.HBox([btn_add, btn_clear]),
    widgets.HTML("<hr>"),
    chk_peaks, chk_fill,
    out_status, out_plot
]))

**Experiments:**
1. Add a **−20 kN midspan load** → triangular SFD, parabolic BMD peak at midspan.
2. Add a **−5 kN/m UDL** from 2 to 8 m → compare BMD shape.
3. Add a **−10 kN load on the left cantilever** → a hogging (negative) zone appears at the left end.
4. Move support A to x=0 → left cantilever disappears. Does M[0] still equal zero?
5. **Balanced cantilever:** add equal loads at both tips → observe that the midspan moment decreases!

---

## 6.5 Practice Exercise

For a beam with L=12 m, x_A=3 m, x_B=9 m, loaded with a −10 kN/m UDL over the full span:
1. Use `beam_solver.analyse()` (the one-call version) to get all results
2. Find the x-position of maximum moment using `x[M.argmax()]`
3. Verify that M at x=0 and x=12 m is ≈ 0

In [ ]:
# Your code here


---

## Summary

**Python concepts covered:**
- `scipy.integrate.cumulative_trapezoid` — step-by-step numerical integration
- `ax.fill_between()` with `where=` — region colouring
- `fig.add_gridspec()` — controlling subplot sizes
- `np.where(np.diff(np.sign(V)))` — locating sign changes
- `fig.suptitle()` — title spanning multiple subplots

**Structural concepts covered:**
- BMD is the integral of SFD: dM/dx = V
- Sagging (+) vs hogging (−) sign convention
- Zero moment at free ends, NOT necessarily at supports
- Peak moment location always where V = 0
- Balanced cantilever effect: cantilever loads reduce midspan moment

## Before the Next Lesson

In the **Capstone (Lesson 7)** you'll wrap everything into a professional tabbed interface, save beam configurations to a file, and export a PDF analysis report — just like you'd produce for a client.